<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de Lenguaje Natural
# Pre-training OPT

En este código entrenamos desde cero un modelo de lenguaje OPT usando el dataset Tiny Shakespeare, pasando por todo el pipeline: tokenización, creación de batches, configuración del modelo y entrenamiento. Luego utilizamos el modelo entrenado para generar texto nuevo a partir de un prompt, controlando la creatividad mediante técnicas de sampling.

Importamos lo necesario

In [ ]:
import torch
from datasets import Dataset
# Herramientas de Hugging Face para modelos de lenguaje
from transformers import (
    AutoTokenizer,                    # Tokenizador automático (convierte texto a números)
    OPTConfig,                        # Configuración del modelo OPT
    OPTForCausalLM,                   # Modelo OPT para generación de texto (causal language model)
    DataCollatorForLanguageModeling,  # Prepara los datos para entrenamiento (batching + máscaras)
    TrainingArguments,                # Configuración del entrenamiento
    Trainer                          # Clase que maneja el entrenamiento automáticamente
)
from itertools import chain

Detectamos dispositivo

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando dispositivo:", device)

Usando dispositivo: cuda


Descargamos el dataset de Tiny Shakespeare

In [ ]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

Convertimos el texto en líneas (dataset simple)

In [ ]:
lines = text.split("\n")
dataset = Dataset.from_dict({"text": lines})

print("Ejemplo:", dataset[0])

Ejemplo: {'text': 'First Citizen:'}


Tokenizamos (reutilizamos OPT)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-125m")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

OPT no tiene pad_token por defecto

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

Definimos una función para realizar tokenización con chunking eficiente

In [ ]:
def chunkify(examples):
    """
    Convierte texto a tokens, concatena y divide en chunks fijos
    """

    # Tokenizamos
    tokens = tokenizer(examples["text"])

    # Aplanamos la lista de listas
    concat = list(chain(*tokens["input_ids"]))

    chunk_size = 256

    # Creamos bloques de tamaño fijo
    chunks = [
        concat[i:i+chunk_size]
        for i in range(0, len(concat) - chunk_size + 1, chunk_size)
    ]

    # Attention mask (todo 1 porque no hay padding interno)
    attention_mask = [[1] * chunk_size for _ in chunks]

    return {
        "input_ids": chunks,
        "attention_mask": attention_mask
    }

Aplicamos transformación

In [ ]:
tokenized = dataset.map(
    chunkify,
    batched=True,
    remove_columns=["text"]
)


print("Ejemplo tokenizado:", tokenized[0])

Map:   0%|          | 0/40001 [00:00<?, ? examples/s]

Ejemplo tokenizado: {'input_ids': [2, 10993, 16436, 35, 2, 17206, 52, 9073, 143, 617, 6, 1798, 162, 1994, 4, 2, 2, 3684, 35, 2, 29235, 677, 6, 1994, 4, 2, 2, 10993, 16436, 35, 2, 1185, 32, 70, 8179, 1195, 7, 1597, 87, 7, 13403, 1173, 116, 2, 2, 3684, 35, 2, 20028, 19084, 4, 8179, 4, 2, 2, 10993, 16436, 35, 2, 10993, 6, 47, 216, 230, 1439, 687, 1127, 40271, 16, 834, 8636, 7, 5, 82, 4, 2, 2, 3684, 35, 2, 170, 216, 75, 6, 52, 216, 75, 4, 2, 2, 10993, 16436, 35, 2, 7939, 201, 3549, 123, 6, 8, 52, 581, 33, 7636, 23, 84, 308, 425, 4, 2, 6209, 75, 10, 7035, 116, 2, 2, 3684, 35, 2, 3084, 55, 1686, 15, 75, 131, 905, 24, 28, 626, 35, 409, 6, 409, 328, 2, 2, 32703, 16436, 35, 2, 3762, 2136, 6, 205, 2286, 4, 2, 2, 10993, 16436, 35, 2, 170, 32, 9521, 2129, 2286, 6, 5, 10512, 4063, 2071, 205, 4, 2, 2264, 3446, 8113, 7068, 2629, 15, 74, 21082, 201, 35, 114, 51, 2, 14656, 3363, 201, 53, 5, 44459, 1571, 6, 150, 24, 58, 2, 11613, 7003, 4399, 6, 52, 429, 4443, 51, 15126, 201, 26899, 352, 131, 2, 4297, 51

Realizamos el split de train y validation

In [ ]:
split = tokenized.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

Definimos modelo OPT (desde cero)

In [ ]:
config = OPTConfig(
    vocab_size=len(tokenizer),

    # Longitud máxima
    max_position_embeddings=256,

    # Arquitectura (ligera para Colab)
    hidden_size=512,
    num_hidden_layers=8,
    num_attention_heads=8,
    ffn_dim=2048,

    dropout=0.1
)

model = OPTForCausalLM(config).to(device)

print("Parámetros del modelo:", model.num_parameters())

Parámetros del modelo: 51087872


Definimos Data Collator (Causal LM)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # IMPORTANTE: causal, no masked LM
)

Definimos Training Arguments (optimizados)

In [ ]:
training_args = TrainingArguments(
    output_dir="./tiny_opt_pretrain",

    # Batch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    # Entrenamiento
    num_train_epochs=5,   # número de veces que el modelo ve todo el dataset
    learning_rate=5e-4,   # CLAVE para pretraining desde cero
    warmup_steps=200,

    # Regularización
    weight_decay=0.01, # penaliza pesos grandes → modelo más generalizable

    # Logging y evaluación
    logging_steps=50, # cada cuántos pasos imprime métricas (loss, etc.)
    save_steps=200, # cada cuántos pasos guarda el modelo

    # Performance
    fp16=True if device == "cuda" else False,

    # Otros
    report_to="none"  # Desactiva reportes a herramientas externas (wandb, etc.)
)

Definimos el trainer

In [ ]:
trainer = Trainer(
    model=model,                 # El modelo que queremos entrenar (OPT)
    args=training_args,          # Configuración del entrenamiento (learning rate, epochs, batch, etc.)
    train_dataset=train_dataset, # Dataset de entrenamiento (datos que el modelo usa para aprender)
    eval_dataset=eval_dataset,   # Dataset de evaluación (para medir qué tan bien está aprendiendo)
    data_collator=data_collator  # Función que prepara los batches (padding, labels, formato correcto)
)

Entrenamos

In [ ]:
trainer.train()

Step,Training Loss
50,9.671077
100,7.066678
150,6.355682
200,5.937812
250,5.710250
300,5.452515
350,5.262917
400,5.171917
450,5.099066
500,4.908989


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=735, training_loss=5.653307482661033, metrics={'train_runtime': 119.4597, 'train_samples_per_second': 49.012, 'train_steps_per_second': 6.153, 'total_flos': 226811384954880.0, 'train_loss': 5.653307482661033, 'epoch': 5.0})

Generamos texto

In [ ]:
model.eval()

prompt = "ROMEO:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

output = model.generate(
    **inputs,
    max_length=200,

    # Sampling (mejor calidad)
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.8,

    pad_token_id=tokenizer.eos_token_id
)

print("\n=== GENERACIÓN ===\n")
print(tokenizer.decode(output[0], skip_special_tokens=True))


=== GENERACIÓN ===

ROMEO: is a man, is, my son, I have, it is a good man! You: and my lord? O, there; 'tis. Come, or I will, do, what I will take your brother, I, and I do you, sir, if you. why, where is not be, I will not. I think: if you, and my tongue. I pray you, and 'tis a day! I will I'll swear that, my father, good! poor. I have not so. I shall go again. good; 'tis my lord. I will. I am. man, sir, I can go. where, good,--O, how, my heart. cousin's good, my name, I love. I'll be: give not not: I will. good good. I come, you? I do you with it! we are, as I will? I love, a man: you!
